# Check the shared rules candidate

Run All to execute the candidate rule checks locally, with progress and a saved result. No training or random-game batch starts here. The new tracing checks compare tracing enabled/disabled and inspect play, recruitment, Reborn, transformation, choices and rollback. Passing checks validate our implementation, not independent Hearthstone fidelity.


In [ ]:
from pathlib import Path
import json, subprocess, sys, uuid
from IPython.display import display, HTML

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'staging/rebased-88/expanded/status.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside death-knight-lab.')
CANDIDATE = ROOT / 'staging/rebased-88'
LOG_DIR = CANDIDATE / 'runs/expanded_validation'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / ('notebook-checks-' + uuid.uuid4().hex + '.log')
print('Ready to check the candidate in a fresh Python process.')


In [ ]:
script = """import json
from expanded.status import run_rule_fixtures
def progress(done,total,failures,errors):
    print(json.dumps(dict(progress=True,done=done,total=total,failures=failures,errors=errors)),flush=True)
r=run_rule_fixtures(progress=progress)
print(json.dumps(dict(result=r)),flush=True)
raise SystemExit(not r['success'])
"""
bar = display(HTML('<progress max="1" value="0"></progress> Starting checks…'), display_id=True)
result = None
with LOG.open('w') as log:
    process = subprocess.Popen([sys.executable, '-u', '-c', script], cwd=CANDIDATE,
                               stdout=subprocess.PIPE, stderr=log, text=True)
    try:
        for line in process.stdout:
            try:
                item = json.loads(line)
            except ValueError:
                print(line.rstrip())
                continue
            if item.get('progress'):
                done, total = item['done'], item['total']
                bar.update(HTML(f'<progress max="{total}" value="{done}"></progress> '
                                f'{done}/{total} checks; {item["failures"]} failures; {item["errors"]} errors'))
            if 'result' in item:
                result = item['result']
        code = process.wait()
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
if result:
    print('Passed:', result['success'])
    print('Checks run:', result['tests_run'])
    print('Saved result:', result['report_path'])
print('Detailed log:', LOG)
if code or result is None:
    print('\n'.join(LOG.read_text().splitlines()[-80:]))
    raise RuntimeError('Checks did not pass. Paste the result and error details back.')
